In [1]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd

from src.utils.helper import startup_investments_engine_pyspark

from src.staging.extract.extract_db import extract_database
from src.staging.extract.extract_db_pyspark import extract_database as extract_database_pyspark
from src.staging.extract.extract_spreadsheet_pyspark import extract_sheet_spark,extract_spreadsheet as extract_spreadsheet_pyspark
from src.staging.extract.extract_api_pyspark import extract_api_milestones_spark,extract_api_spark,extract_backfilling_spark,extract_api_milestones_spark

from src.staging.extract.extract_spreadsheet import extract_spreadsheet
from src.staging.extract.extract_api import extract_api_milestones,extract_backfilling
from src.staging.load.load import load_staging
from src.staging.extract.extract_spreadsheet import extract_spreadsheet, extract_sheet

from src.warehouse.extract.extract_db import extract_database as extract_staging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from src.staging.load.load_pyspark import load_staging_pyspark_upsert
from datetime import datetime
from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging


# Declare spark

In [2]:
from pyspark.sql import SparkSession

# create spark session
spark = SparkSession.builder \
    .appName("Pipeline Staging") \
    .config("spark.ui.enabled", "true") \
    .getOrCreate()

print(spark.version)  # Menampilkan versi Spark


3.3.2


# Staging

## Extract

### api

In [8]:
df_staging_api = extract_api_milestones_spark(spark, table_name='milestones')

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\sql\pandas\conversion.py:474: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\sql\pandas\conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\sql\pandas\conversion.py:474: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\sql\pandas\conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteri

+-------+----------+-------+------+----------+-------------------+---------+
|   step|   process| status|source|table_name|           etl_date|error_msg|
+-------+----------+-------+------+----------+-------------------+---------+
|staging|extraction|success|   api|milestones|2025-04-25 09:52:11|     null|
+-------+----------+-------+------+----------+-------------------+---------+



### spreadsheet

In [3]:
# relationship 
people_df = extract_spreadsheet_pyspark(spark,table_name='people')
people_df

# people 
relationships_df = extract_spreadsheet_pyspark(spark,table_name='relationships')
relationships_df


+-------+----------+-------+-----------+----------+-------------------+---------+
|   step|   process| status|     source|table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+----------+-------------------+---------+
|staging|extraction|success|spreadsheet|    people|2025-04-25 10:29:41|     null|
+-------+----------+-------+-----------+----------+-------------------+---------+

+-------+----------+-------+-----------+-------------+-------------------+---------+
|   step|   process| status|     source|   table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+-------------+-------------------+---------+
|staging|extraction|success|spreadsheet|relationships|2025-04-25 10:30:09|     null|
+-------+----------+-------+-----------+-------------+-------------------+---------+



DataFrame[relationship_id: string, person_object_id: string, relationship_object_id: string, start_at: string, end_at: string, is_past: string, sequence: string, title: string, created_at: timestamp, updated_at: string]

### DB

In [4]:
# acquisition

acquisition = extract_database_pyspark(spark,'acquisition')

#company
company = extract_database_pyspark(spark,'company')

#funding_rounds
funding_rounds = extract_database_pyspark(spark,'funding_rounds')

#funds
funds = extract_database_pyspark(spark,'funds')

#investments
investments = extract_database_pyspark(spark,'investments')

#ipos
ipos = extract_database_pyspark(spark,'ipos')


+-------+----------+-------+--------+-----------+-------------------+---------+
|   step|   process| status|  source| table_name|           etl_date|error_msg|
+-------+----------+-------+--------+-----------+-------------------+---------+
|staging|extraction|success|database|acquisition|2025-04-25 09:46:55|     null|
+-------+----------+-------+--------+-----------+-------------------+---------+

+-------+----------+-------+--------+----------+-------------------+---------+
|   step|   process| status|  source|table_name|           etl_date|error_msg|
+-------+----------+-------+--------+----------+-------------------+---------+
|staging|extraction|success|database|   company|2025-04-25 09:47:05|     null|
+-------+----------+-------+--------+----------+-------------------+---------+

+-------+----------+-------+--------+--------------+-------------------+---------+
|   step|   process| status|  source|    table_name|           etl_date|error_msg|
+-------+----------+-------+--------+

## Load

### spreadsheet

In [6]:
load_staging_pyspark_upsert(spark, data=people_df, schema='public', table_name='people', idx_name='people_id', source='spreadsheet')
load_staging_pyspark_upsert(spark, data=relationships_df, schema='public', table_name='relationships', idx_name='relationship_id', source='spreadsheet')


+-------+-------+-------+--------+----------+--------------------+---------+
|   step|process| status|  source|table_name|            etl_date|error_msg|
+-------+-------+-------+--------+----------+--------------------+---------+
|staging|   load|success|database|    people|2025-04-25 10:32:...|     null|
+-------+-------+-------+--------+----------+--------------------+---------+

+-------+-------+-------+--------+-------------+--------------------+---------+
|   step|process| status|  source|   table_name|            etl_date|error_msg|
+-------+-------+-------+--------+-------------+--------------------+---------+
|staging|   load|success|database|relationships|2025-04-25 10:32:...|     null|
+-------+-------+-------+--------+-------------+--------------------+---------+



### api

In [9]:
load_staging_pyspark_upsert(spark, data=df_staging_api, schema='public', table_name='milestones', idx_name='milestone_id', source='api')


+-------+-------+-------+--------+----------+--------------------+---------+
|   step|process| status|  source|table_name|            etl_date|error_msg|
+-------+-------+-------+--------+----------+--------------------+---------+
|staging|   load|success|database|milestones|2025-04-25 09:59:...|     null|
+-------+-------+-------+--------+----------+--------------------+---------+



### DB

In [10]:
# acquisition
load_staging_pyspark_upsert(spark, data=acquisition, schema='public', table_name='acquisition', idx_name='acquisition_id', source='database')
#company
load_staging_pyspark_upsert(spark, data=company, schema='public', table_name='company', idx_name='object_id', source='database')

#funding_rounds
load_staging_pyspark_upsert(spark, data=funding_rounds, schema='public', table_name='funding_rounds', idx_name='funding_round_id', source='database')

#funds
load_staging_pyspark_upsert(spark, data=funds, schema='public', table_name='funds', idx_name='fund_id', source='database')

#investments
load_staging_pyspark_upsert(spark, data=investments, schema='public', table_name='investments', idx_name='investment_id', source='database')

#ipos
load_staging_pyspark_upsert(spark, data=ipos, schema='public', table_name='ipos', idx_name='ipo_id', source='database')


+-------+-------+-------+--------+-----------+--------------------+---------+
|   step|process| status|  source| table_name|            etl_date|error_msg|
+-------+-------+-------+--------+-----------+--------------------+---------+
|staging|   load|success|database|acquisition|2025-04-25 10:02:...|     null|
+-------+-------+-------+--------+-----------+--------------------+---------+

+-------+-------+-------+--------+----------+--------------------+---------+
|   step|process| status|  source|table_name|            etl_date|error_msg|
+-------+-------+-------+--------+----------+--------------------+---------+
|staging|   load|success|database|   company|2025-04-25 10:02:...|     null|
+-------+-------+-------+--------+----------+--------------------+---------+

+-------+-------+-------+--------+--------------+--------------------+---------+
|   step|process| status|  source|    table_name|            etl_date|error_msg|
+-------+-------+-------+--------+--------------+------------

# Warehouse

## Extract

### api

In [12]:
milestones_staging = extract_db_pyspark_staging(spark,table_name='milestones')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|milestones|2025-04-25 10:04:17|     null|
+---------+----------+-------+--------+----------+-------------------+---------+



### spreadsheet

In [8]:
# relationship 
people_staging = extract_db_pyspark_staging(spark,table_name='people')

# people 
relationships_staging = extract_db_pyspark_staging(spark,table_name='relationships')


+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|    people|2025-04-25 10:34:12|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+----------+-------+--------+-------------+-------------------+---------+
|     step|   process| status|  source|   table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-------------+-------------------+---------+
|warehouse|extraction|success|database|relationships|2025-04-25 10:34:22|     null|
+---------+----------+-------+--------+-------------+-------------------+---------+



### DB

In [11]:
# acquisition

acquisition_staging = extract_db_pyspark_staging(spark,'acquisition')

#company
company_staging = extract_db_pyspark_staging(spark,'company')

#funding_rounds
funding_rounds_staging = extract_db_pyspark_staging(spark,'funding_rounds')

#funds
funds_staging = extract_db_pyspark_staging(spark,'funds')

#investments
investments_staging = extract_db_pyspark_staging(spark,'investments')

#ipos
ipos_staging = extract_db_pyspark_staging(spark,'ipos')


+---------+----------+-------+--------+-----------+-------------------+---------+
|     step|   process| status|  source| table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-----------+-------------------+---------+
|warehouse|extraction|success|database|acquisition|2025-04-25 10:35:23|     null|
+---------+----------+-------+--------+-----------+-------------------+---------+

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|   company|2025-04-25 10:35:33|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+----------+-------+--------+--------------+-------------------+---------+
|     step|   process| status|  source|    table_name|           etl_date|error_msg|
+---------+--

### transform

In [25]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd


from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging

from src.warehouse.transform.dim_company_pyspark import transform_dim_company_spark
from src.warehouse.transform.dim_people_pyspark import transform_dim_people_spark
from src.warehouse.transform.dim_relationship_pyspark import transform_dim_relationship_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_funding_rounds_pyspark import transform_fact_funding_rounds_spark
from src.warehouse.transform.fact_funds_pyspark import transform_fact_funds_spark
from src.warehouse.transform.fact_ipos_pyspark import transform_fact_ipos_spark
from src.warehouse.transform.fact_milestones_pyspark import transform_fact_milestones_spark
from src.warehouse.transform.fact_investments_pyspark import transform_fact_investments_spark
# from src.warehouse.transform.dim_relationship import transform_dim_relationship
# from src.warehouse.transform.fact_acquisitions import transform_fact_acquisitions
# from src.warehouse.transform.fact_funding_rounds import transform_fact_funding_rounds
# from src.warehouse.transform.fact_funds import transform_fact_funds
# from src.warehouse.transform.fact_investments import transform_fact_investments
# from src.warehouse.transform.fact_ipos import transform_fact_ipos
# from src.warehouse.transform.fact_milestones import transform_fact_milestones

from src.warehouse.load.load import load_warehouse
from src.warehouse.load.load_pyspark import load_warehouse_pyspark_upsert


from pyspark.sql import SparkSession

# create spark session
spark = SparkSession.builder \
    .appName("Pipeline Staging") \
    .config("spark.ui.enabled", "true") \
    .getOrCreate()

print(spark.version)  # Menampilkan versi Spark


3.3.2


In [27]:
# relationship 
# people_staging = extract_db_pyspark_staging(spark,table_name='people')

# people 
# relationships_staging = extract_db_pyspark_staging(spark,table_name='relationships')


# acquisition

# acquisition_staging = extract_db_pyspark_staging(spark,'acquisition')

# #company
# company_staging = extract_db_pyspark_staging(spark,'company')

# #funding_rounds
funding_rounds_staging = extract_db_pyspark_staging(spark,'funding_rounds')

#funds
# funds_staging = extract_db_pyspark_staging(spark,'funds')

# #investments
# investments_staging = extract_db_pyspark_staging(spark,'investments')

# #ipos
# ipos_staging = extract_db_pyspark_staging(spark,'ipos')


# #ipos
# milestones_staging = extract_db_pyspark_staging(spark,'milestones')

+---------+----------+-------+--------+--------------+-------------------+---------+
|     step|   process| status|  source|    table_name|           etl_date|error_msg|
+---------+----------+-------+--------+--------------+-------------------+---------+
|warehouse|extraction|success|database|funding_rounds|2025-04-25 16:09:32|     null|
+---------+----------+-------+--------+--------------+-------------------+---------+



In [28]:

# # # dim_company
# dim_company = transform_dim_company_spark(spark,company_staging,'company')

# # dim_company
# dim_people = transform_dim_people_spark(spark, people_staging,'people')

# dim_company
# dim_relationships = transform_dim_relationship_spark(spark,relationships_staging,'relationships')
# # dim_company
# dim_company = transform_dim_company(company,'dim_company')
# dim_people
# fact_acquisitions = transform_fact_acquisitions_spark(spark,acquisition_staging,'acquisition')
# fact_funds_spark = transform_fact_funds_spark(spark,funds_staging,'funds')

fact_funding_rounds_spark = transform_fact_funding_rounds_spark(spark,funding_rounds_staging,'funding_rounds')
# fact_ipos = transform_fact_ipos_spark(spark,ipos_staging,'ipos')
# fact_milestones = transform_fact_milestones_spark(spark,milestones_staging,'milestones')
# fact_investments = transform_fact_investments_spark(spark,investments_staging,'investments')


+---------+--------------+-------+-------+--------------+-------------------+---------+
|     step|       process| status| source|    table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+--------------+-------------------+---------+
|warehouse|transformation|success|staging|funding_rounds|2025-04-25 16:10:11|     null|
+---------+--------------+-------+-------+--------------+-------------------+---------+



In [35]:
fact_funding_rounds_spark.printSchema()


root
 |-- funding_round_nk: integer (nullable = true)
 |-- funded_at: integer (nullable = true)
 |-- funding_round_type: string (nullable = true)
 |-- funding_round_code: string (nullable = true)
 |-- raised_amount_usd: decimal(15,2) (nullable = false)
 |-- pre_money_valuation_usd: decimal(15,2) (nullable = false)
 |-- post_money_valuation_usd: decimal(15,2) (nullable = false)
 |-- round_position_desc: string (nullable = false)
 |-- round_stage_desc: string (nullable = false)
 |-- company_id: string (nullable = true)



In [32]:
fact_funding_rounds_spark.show()

+----------------+---------+------------------+------------------+-----------------+-----------------------+------------------------+-------------------+----------------+--------------------+
|funding_round_nk|funded_at|funding_round_type|funding_round_code|raised_amount_usd|pre_money_valuation_usd|post_money_valuation_usd|round_position_desc|round_stage_desc|          company_id|
+----------------+---------+------------------+------------------+-----------------+-----------------------+------------------------+-------------------+----------------+--------------------+
|               3| 20050501|          series-a|                 a|      12700000.00|           115000000.00|                    0.00|    Not First Round|   Ongoing Round|c285db96-69bd-4db...|
|               6| 20070101|          series-a|                 a|       1500000.00|             8500000.00|             10000000.00|    Not First Round|   Ongoing Round|e41c4b8b-e297-414...|
|              12| 20070601|          se

## Load

In [34]:
# load_warehouse_pyspark_upsert(spark=spark,data=dim_company, table_name='dim_company', schema='public', 
#                idx_name='company_nk', source='staging',table_process='company')

# load_warehouse_pyspark_upsert(spark=spark,data=dim_people, table_name='dim_people', schema='public', 
#                idx_name='people_nk', source='staging',table_process='people')

load_warehouse_pyspark_upsert(spark=spark,data=fact_funding_rounds_spark, table_name='fact_funding_rounds', schema='public', 
               idx_name='funding_round_nk', source='staging',table_process='funding_rounds')


Load failed: An error occurred while calling o2136.jdbc.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 165.0 failed 1 times, most recent failure: Lost task 1.0 in stage 165.0 (TID 250) (DESKTOP-E0SUGV9.mshome.net executor driver): java.sql.BatchUpdateException: Batch entry 0 INSERT INTO public.fact_funding_rounds ("funding_round_nk","funded_at","funding_round_type","funding_round_code","raised_amount_usd","pre_money_valuation_usd","post_money_valuation_usd","round_position_desc","round_stage_desc","company_id") VALUES (('2'::int4),('20040901'::int4),('angel'),('angel'),('500000.00'::numeric),('0.00'::numeric),('0.00'::numeric),('Not First Round'),('Ongoing Round'),('c285db96-69bd-4db4-a122-25a229deea0d')) was aborted: ERROR: column "company_id" is of type uuid but expression is of type character varying
  Hint: You will need to rewrite or cast the expression.
  Position: 277  Call getNextException to see other errors in the batch.
	at org.postgres

### spreadsheet

In [ ]:
load_staging(data=people_df, table_name='people', schema='public', idx_name='people_id', source='spreadsheet')
load_staging(data=relationships_df, table_name='relationships', schema='public', idx_name='relationship_id', source='spreadsheet')

### api

In [ ]:
load_staging(data=df_staging_api, table_name='milestones', schema='public', idx_name='milestone_id', source='api')


### DB

In [ ]:
# acquisition
load_staging(data=acquisition, table_name='acquisition', schema='public', idx_name='acquisition_id', source='database')
#company
load_staging(data=company, table_name='company', schema='public', idx_name='object_id', source='database')

#funding_rounds
load_staging(data=funding_rounds, table_name='funding_rounds', schema='public', idx_name='funding_round_id', source='database')

#funds
load_staging(data=funds, table_name='funds', schema='public', idx_name='fund_id', source='database')

#investments
load_staging(data=investments, table_name='investments', schema='public', idx_name='investment_id', source='database')

#ipos
load_staging(data=ipos, table_name='ipos', schema='public', idx_name='ipo_id', source='database')
